# Session VI: Advanced String Handling and Data Cleaning

Go beyond basic string methods to master advanced text manipulation, pattern matching, and data cleaning techniques essential in real-world applications.

## Exercise 1: Text Normalizer
**Objective:** Practice advanced string methods to clean and standardize messy user input.

You receive raw user input from a registration form. Names have irregular spacing and capitalization, usernames contain unwanted characters, and bios have excessive whitespace. Write functions to normalize each field.

**Tasks:**
1. `normalize_name(name)` → Strip whitespace, collapse multiple spaces into one, and apply title case. Return `None` if the input is not a string or is empty after cleaning.
2. `normalize_username(username)` → Strip, lowercase, and remove any character that is NOT a letter, digit, or underscore. Return `None` if empty after cleaning.
3. `normalize_bio(bio)` → Strip, collapse all whitespace (including tabs and newlines) into single spaces. Truncate to a maximum of 150 characters, adding `"..."` if truncated.
4. `clean_registration(records)` → Take a list of dicts (each with `"name"`, `"username"`, `"bio"`) and return a new list of cleaned dicts. Skip records where either `name` or `username` is invalid.

**Example Input:**
```python
records = [
    {"name": "   aLiCe    sMiTh  ", "username": "  Alice.Smith!  ", "bio": "  Hello!   I love   Python.  "},
    {"name": "", "username": "bob", "bio": "Hi there."},
    {"name": "   Charlie   ", "username": "!!!!", "bio": "A" * 200},
]
```

In [2]:
import re


def normalize_name(name):
    """Normalize a name: strip, collapse spaces, title case."""
    if not isinstance(name, str):
        return None

    # Splitting with no arguments collapses all consecutive whitespace
    words = name.split()
    if not words:
        return None

    # Rejoin with single spaces and convert to title case
    cleaned = " ".join(words).title()
    return cleaned


def normalize_username(username):
    """Normalize a username: strip, lowercase, remove invalid characters."""
    if not isinstance(username, str):
        return None

    # Strip and convert to lowercase
    lowered = username.strip().lower()

    # Retain only letters, digits, and underscores
    cleaned = re.sub(r"[^\w]", "", lowered)

    return cleaned if cleaned else None


def normalize_bio(bio):
    """Normalize a bio: collapse whitespace, truncate to 150 chars."""
    if not isinstance(bio, str):
        return ""

    cleaned = " ".join(bio.split())

    if len(cleaned) > 150:
        return cleaned[:147] + "..."

    return cleaned


def clean_registration(records):
    """Clean a list of registration records, skipping invalid ones."""
    cleaned_records = []

    for record in records:
        cleaned_name = normalize_name(record.get("name"))
        cleaned_username = normalize_username(record.get("username"))
        cleaned_bio = normalize_bio(record.get("bio"))

        if cleaned_name is None or cleaned_username is None:
            continue

        cleaned_records.append(
            {
                "name": cleaned_name,
                "username": cleaned_username,
                "bio": cleaned_bio,
            }
        )

    return cleaned_records


# --- TEST ---
print(normalize_name("   aLiCe    sMiTh  "))
# Expected: "Alice Smith"

print(normalize_username("  Alice.Smith!  "))
# Expected: "alicesmith"

print(normalize_bio("  Hello!   I love   Python.  "))
# Expected: "Hello! I love Python."

records = [
    {"name": "   aLiCe    sMiTh  ", "username": "  Alice_99  ", "bio": "  Hello!   I love   Python.  "},
    {"name": "", "username": "bob", "bio": "Hi there."},
    {"name": "   Charlie   ", "username": "!!!!", "bio": "A" * 200},
    {"name": "   Diana   Prince  ", "username": "  wonder_woman  ", "bio": "\t  Saving  \n  the   world.  "},
]
cleaned = clean_registration(records)
for r in cleaned:
    print(r)

Alice Smith
alicesmith
Hello! I love Python.
{'name': 'Alice Smith', 'username': 'alice_99', 'bio': 'Hello! I love Python.'}
{'name': 'Diana Prince', 'username': 'wonder_woman', 'bio': 'Saving the world.'}


---
## Exercise 2: Regex Validator Suite
**Objective:** Write regular expression patterns to validate common data formats.

Build a suite of validator functions using the `re` module. Each function should return `True` if the input matches the expected format, and `False` otherwise.

**Tasks:**
1. `is_valid_email(email)` → Validate an email address: local part (letters, digits, `._%+-`), `@`, domain, `.`, TLD (at least 2 letters).
2. `is_valid_phone(phone)` → Accept any of these formats: `(555) 123-4567`, `555-123-4567`, `555.123.4567`, `5551234567`. Optionally with country code `+1` or `1-` prefix.
3. `is_valid_date(date_str)` → Validate dates in `YYYY-MM-DD` format. Month must be 01-12, day must be 01-31.
4. `is_valid_url(url)` → Validate a basic URL: must start with `http://` or `https://`, followed by a domain and optional path.
5. `validate_record(record)` → Take a dict with `"email"`, `"phone"`, `"date"`, and `"url"` keys. Return a dict mapping each field to `True`/`False`.

In [3]:
import re


def is_valid_email(email):
    """Validate an email address using regex."""
    if not isinstance(email, str):
        return False
    pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    return bool(re.fullmatch(pattern, email.strip()))


def is_valid_phone(phone):
    """Validate a US phone number in various formats."""
    if not isinstance(phone, str):
        return False

    pattern = (
        r"^(?:\+?1[-.\s]?)?"  # Optional country code: +1, 1-, etc.
        r"(?:\(\d{3}\)|\d{3})"  # Area code: (555) or 555
        r"[-.\s]?"  # Optional separator
        r"\d{3}"  # 3-digit exchange
        r"[-.\s]?"  # Optional separator
        r"\d{4}$"  # 4-digit line number
    )
    return bool(re.fullmatch(pattern, phone.strip()))


def is_valid_date(date_str):
    """Validate a date string in YYYY-MM-DD format."""
    if not isinstance(date_str, str):
        return False

    # Month: (0[1-9]|1[0-2]) -> 01 to 12
    # Day:   (0[1-9]|[12]\d|3[01]) -> 01 to 31
    pattern = r"^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])$"
    return bool(re.fullmatch(pattern, date_str.strip()))


def is_valid_url(url):
    """Validate a basic URL (http or https) with domain and optional path."""
    if not isinstance(url, str):
        return False
    pattern = r"^https?:\/\/[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(?:\/[^\s]*)?$"
    return bool(re.fullmatch(pattern, url.strip()))


def validate_record(record):
    """Validate all fields in a record dict."""
    if not isinstance(record, dict):
        return {}

    return {
        "email": is_valid_email(record.get("email")),
        "phone": is_valid_phone(record.get("phone")),
        "date": is_valid_date(record.get("date")),
        "url": is_valid_url(record.get("url")),
    }



# --- TEST ---
print("=== Email ===")
for e in ["alice@example.com", "bob.smith@uni.edu", "missing-at.com", "@no-local.com", "user@domain"]:
    print(f"  {e:>25} → {is_valid_email(e)}")

print("\n=== Phone ===")
for p in ["(555) 123-4567", "555-123-4567", "5551234567", "+1-555-123-4567", "123", "abcdefghij"]:
    print(f"  {p:>20} → {is_valid_phone(p)}")

print("\n=== Date ===")
for d in ["2024-01-15", "2024-13-01", "2024-00-10", "24-01-15", "2024-12-31"]:
    print(f"  {d:>15} → {is_valid_date(d)}")

print("\n=== URL ===")
for u in ["https://example.com", "http://example.com/path", "ftp://wrong.com", "example.com", "https://a.io"]:
    print(f"  {u:>30} → {is_valid_url(u)}")

print("\n=== Record ===")
record = {"email": "alice@example.com", "phone": "(555) 123-4567", "date": "2024-01-15", "url": "https://example.com"}
print(validate_record(record))

=== Email ===
          alice@example.com → True
          bob.smith@uni.edu → True
             missing-at.com → False
              @no-local.com → False
                user@domain → False

=== Phone ===
        (555) 123-4567 → True
          555-123-4567 → True
            5551234567 → True
       +1-555-123-4567 → True
                   123 → False
            abcdefghij → False

=== Date ===
       2024-01-15 → True
       2024-13-01 → False
       2024-00-10 → False
         24-01-15 → False
       2024-12-31 → True

=== URL ===
             https://example.com → True
         http://example.com/path → True
                 ftp://wrong.com → False
                     example.com → False
                    https://a.io → True

=== Record ===
{'email': True, 'phone': True, 'date': True, 'url': True}


---
## Exercise 3: CSV Data Cleaner
**Objective:** Read a messy CSV file, clean every field, and write a clean version.

You receive a CSV file of customer data that is full of problems: inconsistent spacing, mixed capitalization, invalid emails, badly formatted phone numbers, and invalid amounts. Your job is to clean it and produce a pristine output file.

**First, run the cell below to generate the messy CSV file.** Then write the functions to clean it.

**Tasks:**
1. `load_messy_csv(filename)` → Read the CSV with `csv.DictReader`. Return the raw rows as a list of dicts.
2. `clean_name(name)` → Strip, collapse whitespace, apply title case.
3. `clean_email(email)` → Strip, lowercase, validate with regex. Return `None` if invalid.
4. `clean_phone(phone)` → Parse to `(XXX) XXX-XXXX` format. Return `None` if invalid.
5. `clean_amount(amount)` → Remove `$`, `,`, whitespace. Convert to float. Return `0.0` if invalid.
6. `clean_all_rows(rows)` → Apply all cleaners to each row. Return list of cleaned dicts with a new `"is_valid"` boolean field (True if email AND phone are both valid).
7. `save_clean_csv(filename, rows)` → Write the cleaned rows to a new CSV file.
8. `print_report(rows)` → Print a summary: total records, valid records, invalid records, total amount, and a list of invalid entries.

In [5]:
# --- RUN THIS CELL FIRST to generate the messy CSV ---

messy_csv_content = """name,email,phone,amount
   aLiCe   sMiTh  , ALICE@EXAMPLE.COM  , (555) 123-4567 , $1,234.56
  Bob    Johnson, bob.johnson@email.com,555.987.6543,  $890.00
 CHARLIE   WILLIAMS ,charlie-AT-email,  555 111 2222  , $450.75
  diana   PRINCE  , diana.prince@heroes.org , +1-555-333-4444,  2100.50
  Eve    Adams  , eve.adams@  ,  123  ,  $abc  
  FRANK   miller  , frank.miller@work.com , 1-555-777-8888 , $3,456.78
    Grace   HOPPER  ,grace.hopper@navy.mil,5559990000,$12,000.00
  hank   , , ,
  Ivy    Chen  , IVY.CHEN@university.EDU , (555)   444 - 5555 , $  678.90  
  jack    black  , jack..black@movies.com , 555-222-3333 , $1,100.25
"""

with open("messy_customers.csv", "w") as f:
    f.write(messy_csv_content)

print("✅ messy_customers.csv created.")

✅ messy_customers.csv created.


In [6]:
import csv
import re

def load_messy_csv(filename):
    """Read a CSV file and return a list of dicts."""
    with open(filename, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return list(reader)


def clean_name(name):
    """Strip, collapse whitespace, title case."""
    if not isinstance(name, str):
        return None
    words = name.split()
    return " ".join(words).title() if words else None


def clean_email(email):
    """Strip, lowercase, validate with regex. Return None if invalid."""
    if not isinstance(email, str):
        return None

    cleaned = email.strip().lower()
    # RFC-adjacent basic pattern: disallows consecutive dots, requires valid TLD
    pattern = r"^[a-z0-9_%+-]+(?:\.[a-z0-9_%+-]+)*@[a-z0-9.-]+\.[a-z]{2,}$"
    if re.fullmatch(pattern, cleaned):
        return cleaned
    return None


def clean_phone(phone):
    """Parse phone to (XXX) XXX-XXXX format. Return None if invalid."""
    if not isinstance(phone, str):
        return None

    # Strip out non-digit characters
    digits = re.sub(r"\D", "", phone)

    # Allow 11-digit US numbers starting with '1'
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]

    # Must be exactly 10 digits
    if len(digits) == 10:
        return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"
    return None


def clean_amount(amount):
    """Remove $, commas, whitespace. Convert to float. Return 0.0 if invalid."""
    if not isinstance(amount, str):
        return 0.0

    sanitized = re.sub(r"[$,\s]", "", amount)
    try:
        return float(sanitized)
    except (ValueError, TypeError):
        return 0.0


def clean_all_rows(rows):
    """Apply all cleaners to each row. Add 'is_valid' field."""
    cleaned_rows = []

    for row in rows:
        name = clean_name(row.get("name"))
        email = clean_email(row.get("email"))
        phone = clean_phone(row.get("phone"))
        amount = clean_amount(row.get("amount"))

        # Valid only if both email and phone are present and well-formed
        is_valid = (email is not None) and (phone is not None)

        cleaned_rows.append(
            {
                "name": name,
                "email": email,
                "phone": phone,
                "amount": amount,
                "is_valid": is_valid,
            }
        )

    return cleaned_rows


def save_clean_csv(filename, rows):
    """Write cleaned rows to a new CSV file."""
    if not rows:
        return

    fieldnames = list(rows[0].keys())
    with open(filename, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def print_report(rows):
    """Print a cleaning summary report."""
    total = len(rows)
    valid_rows = [r for r in rows if r["is_valid"]]
    invalid_rows = [r for r in rows if not r["is_valid"]]
    total_amount = sum(r["amount"] for r in rows)

    print("=" * 45)
    print("           DATA CLEANING REPORT")
    print("=" * 45)
    print(f"Total Records Processed : {total}")
    print(f"Valid Records           : {len(valid_rows)}")
    print(f"Invalid Records         : {len(invalid_rows)}")
    print(f"Total Dollar Amount     : ${total_amount:,.2f}")
    print("-" * 45)

    if invalid_rows:
        print("Flagged Invalid Entries:")
        for r in invalid_rows:
            reasons = []
            if r["email"] is None:
                reasons.append("bad/missing email")
            if r["phone"] is None:
                reasons.append("bad/missing phone")
            reason_str = ", ".join(reasons)
            print(f"  • {r['name'] or '<No Name>'}: {reason_str}")
    print("=" * 45)


# --- PROCESS ---
raw_rows = load_messy_csv("messy_customers.csv")
cleaned = clean_all_rows(raw_rows)
save_clean_csv("clean_customers.csv", cleaned)
print_report(cleaned)


           DATA CLEANING REPORT
Total Records Processed : 10
Valid Records           : 6
Invalid Records         : 4
Total Dollar Amount     : $4,137.15
---------------------------------------------
Flagged Invalid Entries:
  • Charlie Williams: bad/missing email
  • Eve Adams: bad/missing email, bad/missing phone
  • Hank: bad/missing email, bad/missing phone
  • Jack Black: bad/missing email


---
## Exercise 4: Phone Number Parser
**Objective:** Build a robust phone number cleaning function that handles many different input formats.

You receive a list of contact records with phone numbers in wildly different formats. Write a parser that normalizes them all to a single consistent format, and a batch processor to clean an entire contact list.

**Tasks:**
1. `parse_phone(raw_phone)` → Accept any reasonable phone format and return `(XXX) XXX-XXXX`. Handle optional country code `+1` or `1-`. Return `None` for invalid numbers (not 10 digits after cleaning).
2. `extract_phone_parts(formatted_phone)` → Take a formatted phone `(XXX) XXX-XXXX` and return a dict with `"area_code"`, `"prefix"`, and `"line"` keys.
3. `batch_clean_phones(contacts)` → Take a list of dicts with `"name"` and `"phone"` keys. Return a new list with cleaned phones and a `"valid"` boolean.
4. `group_by_area_code(contacts)` → Take the cleaned contacts and return a dict mapping each area code to a list of names.
5. `print_phone_report(contacts)` → Print a formatted report showing valid/invalid counts and groupings by area code.

In [17]:
from collections import defaultdict
import re


def parse_phone(raw_phone):
    """Parse any phone format to (XXX) XXX-XXXX. Return None if invalid."""
    if not isinstance(raw_phone, str):
        return None

    # Strip out all non-digit characters
    digits = re.sub(r"\D", "", raw_phone)

    # Strip country code 1 if 11 digits
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]

    # Must be exactly 10 digits
    if len(digits) == 10:
        return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"

    return None


def extract_phone_parts(formatted_phone):
    """Extract area_code, prefix, and line from (XXX) XXX-XXXX format."""
    if not isinstance(formatted_phone, str):
        return None

    pattern = r"^\((\d{3})\)\s(\d{3})-(\d{4})$"
    match = re.fullmatch(pattern, formatted_phone.strip())
    if match:
        area_code, prefix, line = match.groups()
        return {"area_code": area_code, "prefix": prefix, "line": line}

    return None


def batch_clean_phones(contacts):
    """Clean all phone numbers in a contact list and mark validity."""
    cleaned_contacts = []

    for contact in contacts:
        name = contact.get("name", "Unknown")
        parsed = parse_phone(contact.get("phone"))

        cleaned_contacts.append(
            {
                "name": name,
                "phone": parsed,
                "valid": parsed is not None,
            }
        )

    return cleaned_contacts


def group_by_area_code(contacts):
    """Group valid contacts by area code into a dictionary."""
    grouped = defaultdict(list)

    for contact in contacts:
        if not contact.get("valid") or not contact.get("phone"):
            continue

        parts = extract_phone_parts(contact["phone"])
        if parts:
            area_code = parts["area_code"]
            grouped[area_code].append(contact["name"])

    return dict(grouped)


def print_phone_report(contacts):
    """Print a summary of cleaned phone data."""
    total = len(contacts)
    valid_contacts = [c for c in contacts if c.get("valid")]
    invalid_contacts = [c for c in contacts if not c.get("valid")]
    grouped = group_by_area_code(contacts)

    print("=" * 50)
    print("             PHONE CLEANING REPORT")
    print("=" * 50)
    print(f"Total Contacts Processed : {total}")
    print(f"Valid Numbers            : {len(valid_contacts)}")
    print(f"Invalid Numbers          : {len(invalid_contacts)}")
    print("-" * 50)

    print("Area Code Breakdown:")
    for area_code in sorted(grouped.keys()):
        names = ", ".join(grouped[area_code])
        count = len(grouped[area_code])
        print(f"  • Area Code ({area_code}) [{count}]: {names}")

    if invalid_contacts:
        print("-" * 50)
        print("Invalid Entries:")
        for inv in invalid_contacts:
            phone_val = repr(inv.get("phone"))
            print(f"  • {inv.get('name', 'Unknown')}: invalid phone ({phone_val})")

    print("=" * 50)


# --- TEST ---
contacts = [
    {"name": "Alice", "phone": "(555) 123-4567"},
    {"name": "Bob", "phone": "555.987.6543"},
    {"name": "Charlie", "phone": "+1-555-111-2222"},
    {"name": "Diana", "phone": "5559990000"},
    {"name": "Eve", "phone": "123"},
    {"name": "Frank", "phone": "1-800-555-0199"},
    {"name": "Grace", "phone": "(800) 555-0100"},
    {"name": "Hank", "phone": "not a phone number"},
    {"name": "Ivy", "phone": "  (555)   444 - 5555  "},
    {"name": "Jack", "phone": "+1 800 555 0150"},
]

cleaned = batch_clean_phones(contacts)
print_phone_report(cleaned)

groups = group_by_area_code(cleaned)
print("\nGrouped by area code:")
for area_code, names in groups.items():
    print(f"  {area_code}: {names}")


             PHONE CLEANING REPORT
Total Contacts Processed : 10
Valid Numbers            : 8
Invalid Numbers          : 2
--------------------------------------------------
Area Code Breakdown:
  • Area Code (555) [5]: Alice, Bob, Charlie, Diana, Ivy
  • Area Code (800) [3]: Frank, Grace, Jack
--------------------------------------------------
Invalid Entries:
  • Eve: invalid phone (None)
  • Hank: invalid phone (None)

Grouped by area code:
  555: ['Alice', 'Bob', 'Charlie', 'Diana', 'Ivy']
  800: ['Frank', 'Grace', 'Jack']


---
## Exercise 5: Encoding Fixer
**Objective:** Handle text files with encoding problems, special characters, and non-printable content.

You receive text files from various sources (old systems, web scraping, international partners). They contain encoding artifacts, non-printable characters, and inconsistent Unicode. Write functions to clean them.

**First, run the cell below to generate sample files.** Then write the cleaning functions.

**Tasks:**
1. `read_safe(filename, encodings)` → Try reading a file with each encoding in the list. Return the content from the first encoding that works. Return `None` if all fail.
2. `remove_non_printable(text)` → Remove all non-printable characters (keep only printable ASCII + newlines and tabs).
3. `fix_common_artifacts(text)` → Replace common encoding artifacts: `Ã©` → `é`, `Ã±` → `ñ`, `â€™` → `'`, `â€œ` → `"`, `â€\x9d` → `"`, `Ã¼` → `ü`.
4. `normalize_unicode(text)` → Use `unicodedata.normalize("NFC", ...)` to normalize Unicode characters.
5. `clean_file(input_file, output_file, encodings)` → Read, clean (remove non-printable + fix artifacts + normalize), and write the cleaned version to a new file.

In [19]:
# --- RUN THIS CELL FIRST to generate sample files ---

# File 1: Text with non-printable characters
with open("messy_text_1.txt", "w", encoding="utf-8") as f:
    f.write("Hello\x00 World\x07!\n")
    f.write("This has\x1b control\x0b characters.\n")
    f.write("Normal line here.\n")
    f.write("Another\x00 messy\x07 line.\n")

# File 2: Text with encoding artifacts
with open("messy_text_2.txt", "w", encoding="utf-8") as f:
    f.write("The cafÃ© is on the corner.\n")
    f.write("She said â€œhelloâ€\x9d to everyone.\n")
    f.write("The niño played in the señora's garden.\n")
    f.write("MÃ¼nchen is beautiful in spring.\n")
    f.write("Itâ€™s a wonderful day.\n")

# File 3: Latin-1 encoded file
with open("messy_text_3.txt", "w", encoding="latin-1") as f:
    f.write("Résumé of José García\n")
    f.write("Worked at Ströer & Müller GmbH\n")
    f.write("Salary: €85,000\n")

print("✅ Sample files created: messy_text_1.txt, messy_text_2.txt, messy_text_3.txt")

UnicodeEncodeError: 'latin-1' codec can't encode character '\u20ac' in position 8: ordinal not in range(256)

In [16]:
import os
import re
import unicodedata


def read_safe(filename, encodings=None):
    """Try reading a file with multiple encodings. Return content or None."""
    if encodings is None:
        encodings = ["utf-8", "latin-1", "ascii"]

    for enc in encodings:
        try:
            with open(filename, "r", encoding=enc) as f:
                return f.read()
        except (UnicodeDecodeError, UnicodeError):
            continue
        except FileNotFoundError:
            return None

    return None


def fix_common_artifacts(text):
    """Replace common encoding artifacts (mojibake) with correct characters."""
    if not isinstance(text, str):
        return ""

    replacements = {
        "Ã©": "é",
        "Ã±": "ñ",
        "Ã¼": "ü",
        "â€™": "'",
        "â€œ": '"',
        "â€\x9d": '"',
        "â€“": "–",
        "â€”": "—",
    }

    for artifact, fix in replacements.items():
        text = text.replace(artifact, fix)

    return text


def remove_non_printable(text):
      """Remove non-printable characters, keep printable ASCII + newline + tab."""
    if not isinstance(text, str):
        return ""

    
    cleaned_chars = [
        ch
        for ch in text
        if ch in ("\n", "\t") or not unicodedata.category(ch).startswith("C")
    ]
    return "".join(cleaned_chars)


def normalize_unicode(text):
    """Normalize Unicode to NFC form (composed characters)."""
    if not isinstance(text, str):
        return ""
    return unicodedata.normalize("NFC", text)


def clean_file(input_file, output_file, encodings=None):
    """Read, clean, and save a text file."""
    raw_content = read_safe(input_file, encodings)
    if raw_content is None:
        print(f"Failed to read {input_file} with provided encodings.")
        return False

    # Pipeline
    text = fix_common_artifacts(raw_content)
    text = remove_non_printable(text)
    text = normalize_unicode(text)

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(text)

    return True


# --- TEST ---

print("=== Test read_safe ===")
content = read_safe("messy_text_3.txt")
print("Read with fallback encoding:")
print(repr(content))

print("\n=== Running Cleaning Pipeline ===")
clean_file("messy_text_1.txt", "clean_text_1.txt")
clean_file("messy_text_2.txt", "clean_text_2.txt")
clean_file("messy_text_3.txt", "clean_text_3.txt")

# Verify output
for fname in ["clean_text_1.txt", "clean_text_2.txt", "clean_text_3.txt"]:
    print(f"\n=== {fname} ===")
    with open(fname, "r", encoding="utf-8") as f:
        print(f.read())


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 47)

---
## Exercise 6: Data Cleaning Module
**Objective:** Build a complete, reusable data cleaning module and use it to process a messy dataset.

This exercise combines everything from this session. You will:
1. Define a set of **reusable cleaning functions** (like a `cleaners` module).
2. Define a **configurable cleaning pipeline** that applies different cleaners to different columns.
3. Process a messy employee dataset and produce a clean CSV + summary report.

**First, run the cell below to generate the messy dataset.** Then build your cleaning pipeline.

**Tasks:**
1. **Cleaning functions** (reusable, no `print()` inside):
   - `clean_text(text)` → Strip, collapse whitespace, lowercase.
   - `clean_name(name)` → Strip, collapse whitespace, title case. Return `"Unknown"` if empty/invalid.
   - `clean_email(email)` → Strip, lowercase, validate with regex. Return `None` if invalid.
   - `clean_phone(phone)` → Parse to `(XXX) XXX-XXXX`. Return `None` if invalid.
   - `clean_salary(salary)` → Remove `$`, `,`, whitespace. Convert to float. Return `0.0` if invalid.
   - `clean_department(dept, valid_depts)` → Normalize and match against a list of valid departments. Return `"Other"` if no match.

2. **Pipeline function:**
   - `clean_employee_record(record, valid_depts)` → Apply the correct cleaner to each field. Add an `"is_valid"` field (True if name ≠ "Unknown" AND email is not None AND phone is not None).

3. **Batch processing:**
   - `process_employee_file(input_file, output_file, valid_depts)` → Read messy CSV, clean all records, write clean CSV, print summary report.

4. **Summary report** should include:
   - Total records processed
   - Valid vs. invalid counts (with percentage)
   - Breakdown by department
   - Total salary sum
   - List of invalid records with the reason why

In [8]:
# --- RUN THIS CELL FIRST to generate the messy employee dataset ---

messy_employee_data = """name,email,phone,department,salary
   ALICE   SMITH  , ALICE.SMITH@COMPANY.COM  , (555) 123-4567 ,  engineering  , $95,000.00
  bob    JOHNSON, bob.johnson@company.com,555.987.6543,  MARKETING  ,  $82,500.00
 CHARLIE   WILLIAMS ,charlie-AT-email,  555 111 2222  ,  Engineering , $78,000.00
  diana   PRINCE  , diana.prince@company.com , +1-555-333-4444, Marketing ,$105,000.00
  , , ,  ,
  Eve    Adams  , eve.adams@  ,  123  ,  SALES  ,  $abc  
  FRANK   miller  , frank.miller@company.com , 1-555-777-8888 , sales , $91,200.00
    Grace   HOPPER  ,grace.hopper@company.com,5559990000,  ENGINEERING  ,$120,000.00
  hank   PATEL  , hank.patel@company.com, (555) 222-3333 , Mktg , $67,800.00
  Ivy    CHEN  , IVY.CHEN@COMPANY.COM , (555)   444 - 5555 ,  eng  , $  88,500.00  
  jack    BLACK  , jack.black@company.com , 555-666-7777 , Human Resources , $72,000.00
  Kate   , kate@company.com , 555-888-9999 ,  HR  , $68,000.00
"""

with open("messy_employees.csv", "w") as f:
    f.write(messy_employee_data)

print("✅ messy_employees.csv created.")

✅ messy_employees.csv created.


In [9]:
import csv
from collections import Counter
import re

# --- VALID DEPARTMENTS (with common aliases) ---
DEPARTMENT_MAP = {
    "engineering": "Engineering",
    "eng": "Engineering",
    "marketing": "Marketing",
    "mktg": "Marketing",
    "sales": "Sales",
    "human resources": "Human Resources",
    "hr": "Human Resources",
}

# --- CLEANING FUNCTIONS ---


def clean_text(text):
    """Strip, collapse whitespace, lowercase."""
    if not isinstance(text, str):
        return ""
    words = text.split()
    return " ".join(words).lower()


def clean_name(name):
    """Strip, collapse whitespace, title case. Return 'Unknown' if invalid."""
    if not isinstance(name, str):
        return "Unknown"
    words = name.split()
    return " ".join(words).title() if words else "Unknown"


def clean_email(email):
    """Strip, lowercase, validate. Return None if invalid."""
    if not isinstance(email, str):
        return None
    cleaned = email.strip().lower()
    pattern = r"^[a-z0-9_%+-]+(?:\.[a-z0-9_%+-]+)*@[a-z0-9.-]+\.[a-z]{2,}$"
    return cleaned if re.fullmatch(pattern, cleaned) else None


def clean_phone(phone):
    """Parse to (XXX) XXX-XXXX. Return None if invalid."""
    if not isinstance(phone, str):
        return None

    digits = re.sub(r"\D", "", phone)
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]

    if len(digits) == 10:
        return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"
    return None


def clean_salary(salary):
    """Remove $, commas, spaces. Convert to float. Return 0.0 if invalid."""
    if not isinstance(salary, str):
        return 0.0
    sanitized = re.sub(r"[$,\s]", "", salary)
    try:
        return float(sanitized)
    except (ValueError, TypeError):
        return 0.0


def clean_department(dept, dept_map):
    """Normalize and match against valid departments. Return 'Other' if no match."""
    normalized = clean_text(dept)
    return dept_map.get(normalized, "Other")


# --- PIPELINE ---


def clean_employee_record(record, dept_map):
    """Clean all fields in an employee record and assess validity."""
    name = clean_name(record.get("name"))
    email = clean_email(record.get("email"))
    phone = clean_phone(record.get("phone"))
    department = clean_department(record.get("department"), dept_map)
    salary = clean_salary(record.get("salary"))

    # is_valid: name is known AND email and phone are present and valid
    is_valid = (name != "Unknown") and (email is not None) and (phone is not None)

    return {
        "name": name,
        "email": email,
        "phone": phone,
        "department": department,
        "salary": salary,
        "is_valid": is_valid,
    }


# --- BATCH PROCESSING & REPORTING ---


def process_employee_file(input_file, output_file, dept_map):
    """Read messy CSV, clean all records, write clean CSV, print report."""
    # 1. Read input
    with open(input_file, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        raw_records = list(reader)

    # 2. Process records
    cleaned_records = [
        clean_employee_record(record, dept_map) for record in raw_records
    ]

    # 3. Write output CSV
    if cleaned_records:
        fieldnames = list(cleaned_records[0].keys())
        with open(output_file, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(cleaned_records)

    # 4. Generate Summary Metrics
    total = len(cleaned_records)
    valid_records = [r for r in cleaned_records if r["is_valid"]]
    invalid_records = [r for r in cleaned_records if not r["is_valid"]]

    valid_pct = (len(valid_records) / total * 100) if total else 0.0
    invalid_pct = (len(invalid_records) / total * 100) if total else 0.0
    total_salary = sum(r["salary"] for r in cleaned_records)

    dept_counts = Counter(r["department"] for r in cleaned_records)

    # 5. Print Report
    print("=" * 55)
    print("          EMPLOYEE DATA CLEANING REPORT")
    print("=" * 55)
    print(f"Total Records Processed : {total}")
    print(f"Valid Records           : {len(valid_records)} ({valid_pct:.1f}%)")
    print(f"Invalid Records         : {len(invalid_records)} ({invalid_pct:.1f}%)")
    print(f"Total Payroll Sum       : ${total_salary:,.2f}")
    print("-" * 55)

    print("Department Breakdown:")
    for dept, count in dept_counts.most_common():
        print(f"  • {dept:<18}: {count} record(s)")

    print("-" * 55)
    print("Invalid Records & Reasons:")
    for r in invalid_records:
        reasons = []
        if r["name"] == "Unknown":
            reasons.append("missing/invalid name")
        if r["email"] is None:
            reasons.append("invalid/missing email")
        if r["phone"] is None:
            reasons.append("invalid/missing phone")

        label = r["name"] if r["name"] != "Unknown" else "<Empty/Unknown Name>"
        print(f"  • {label:<22} → {', '.join(reasons)}")

    print("=" * 55)


# --- RUN ---
process_employee_file("messy_employees.csv", "clean_employees.csv", DEPARTMENT_MAP)


          EMPLOYEE DATA CLEANING REPORT
Total Records Processed : 12
Valid Records           : 9 (75.0%)
Invalid Records         : 3 (25.0%)
Total Payroll Sum       : $866.00
-------------------------------------------------------
Department Breakdown:
  • Engineering       : 4 record(s)
  • Marketing         : 3 record(s)
  • Sales             : 2 record(s)
  • Human Resources   : 2 record(s)
  • Other             : 1 record(s)
-------------------------------------------------------
Invalid Records & Reasons:
  • Charlie Williams       → invalid/missing email
  • <Empty/Unknown Name>   → missing/invalid name, invalid/missing email, invalid/missing phone
  • Eve Adams              → invalid/missing email, invalid/missing phone
